In [1]:
import pandas as pd

CAMINHO = "../data/raw/"

pedidos    = pd.read_csv(CAMINHO + "olist_orders_dataset.csv")
avaliacoes = pd.read_csv(CAMINHO + "olist_order_reviews_dataset.csv")
itens      = pd.read_csv(CAMINHO + "olist_order_items_dataset.csv")
clientes   = pd.read_csv(CAMINHO + "olist_customers_dataset.csv")
produtos   = pd.read_csv(CAMINHO + "olist_products_dataset.csv")
vendedores = pd.read_csv(CAMINHO + "olist_sellers_dataset.csv")
pagamentos = pd.read_csv(CAMINHO + "olist_order_payments_dataset.csv")

In [2]:
# Alvo: uma avaliação por pedido (a mais recente); negativa = nota 1 ou 2
avaliacoes["review_answer_timestamp"] = pd.to_datetime(avaliacoes["review_answer_timestamp"])
alvo = (avaliacoes.sort_values("review_answer_timestamp")
        .drop_duplicates("order_id", keep="last")[["order_id", "review_score"]])
alvo["negativa"] = (alvo["review_score"] <= 2).astype(int)

# Converte as datas dos pedidos
datas = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
         "order_delivered_customer_date", "order_estimated_delivery_date"]
for c in datas:
    pedidos[c] = pd.to_datetime(pedidos[c])

# Escopo: pedidos entregues (é o momento em que o modelo será usado)
base = (pedidos[(pedidos["order_status"] == "delivered") & pedidos["order_delivered_customer_date"].notna()]
        .merge(alvo, on="order_id"))

print("pedidos na base:", len(base))
print("taxa de avaliação negativa:", round(base["negativa"].mean() * 100, 1), "%")

pedidos na base: 95824
taxa de avaliação negativa: 12.8 %


In [3]:
# Tempos e prazos (tudo conhecido até a entrega)
base["dias_entrega"]            = (base["order_delivered_customer_date"] - base["order_purchase_timestamp"]).dt.days
base["dias_prometidos"]         = (base["order_estimated_delivery_date"] - base["order_purchase_timestamp"]).dt.days
base["dias_atraso"]             = (base["order_delivered_customer_date"] - base["order_estimated_delivery_date"]).dt.days
base["atrasado"]                = (base["dias_atraso"] > 0).astype(int)
base["horas_aprovacao"]         = (base["order_approved_at"] - base["order_purchase_timestamp"]).dt.total_seconds() / 3600
base["dias_ate_transportadora"] = (base["order_delivered_carrier_date"] - base["order_purchase_timestamp"]).dt.days

# Calendário da compra
base["mes_compra"]        = base["order_purchase_timestamp"].dt.month
base["dia_semana_compra"] = base["order_purchase_timestamp"].dt.dayofweek

C:\Users\igorb\AppData\Local\Temp\ipykernel_23580\3153336331.py:11: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  base["dia_semana_compra"] = base["order_purchase_timestamp"].dt.dayofweek


In [4]:
# Junta cada item com os dados do produto e do vendedor
itens_det = (itens.merge(produtos, on="product_id", how="left")
                  .merge(vendedores, on="seller_id", how="left"))

# Resume por pedido (grão: 1 linha por pedido)
por_pedido = (itens_det.groupby("order_id")
              .agg(n_itens=("order_item_id", "count"),
                   n_produtos=("product_id", "nunique"),
                   n_vendedores=("seller_id", "nunique"),
                   valor_total=("price", "sum"),
                   frete_total=("freight_value", "sum"),
                   peso_total_g=("product_weight_g", "sum"),
                   fotos_media=("product_photos_qty", "mean"),
                   tam_descricao_media=("product_description_lenght", "mean"),
                   categoria=("product_category_name", "first"),
                   estado_vendedor=("seller_state", "first"))
              .reset_index())

base = base.merge(por_pedido, on="order_id", how="left")
base["frete_sobre_valor"] = base["frete_total"] / base["valor_total"]

In [5]:
import os

# Pagamentos resumidos por pedido
pag = (pagamentos.groupby("order_id")
       .agg(n_parcelas=("payment_installments", "max"),
            n_formas_pagamento=("payment_type", "nunique"),
            forma_pagamento=("payment_type", "first"))
       .reset_index())
base = base.merge(pag, on="order_id", how="left")

# Estado do cliente e se vendedor e cliente são do mesmo estado
base = base.merge(clientes[["customer_id", "customer_state"]], on="customer_id", how="left")
base = base.rename(columns={"customer_state": "estado_cliente"})
base["mesmo_estado"] = (base["estado_cliente"] == base["estado_vendedor"]).astype(int)

# Seleciona as colunas do modelo
colunas_modelo = ["dias_entrega", "dias_prometidos", "dias_atraso", "atrasado", "horas_aprovacao",
                  "dias_ate_transportadora", "mes_compra", "dia_semana_compra",
                  "n_itens", "n_produtos", "n_vendedores", "valor_total", "frete_total", "frete_sobre_valor",
                  "peso_total_g", "fotos_media", "tam_descricao_media", "n_parcelas", "n_formas_pagamento",
                  "categoria", "estado_vendedor", "estado_cliente", "forma_pagamento", "mesmo_estado"]

base_modelo = base[["order_id", "order_purchase_timestamp"] + colunas_modelo + ["negativa"]]

os.makedirs("../data/processed", exist_ok=True)
base_modelo.to_csv("../data/processed/base_modelo.csv", index=False)

print("formato:", base_modelo.shape)
print("\nvalores faltantes:")
print(base_modelo.isna().sum()[lambda s: s > 0])

formato: (95824, 27)

valores faltantes:
horas_aprovacao              14
dias_ate_transportadora       1
fotos_media                1324
tam_descricao_media        1324
n_parcelas                    1
n_formas_pagamento            1
categoria                  1324
forma_pagamento               1
dtype: int64
